<a href="https://colab.research.google.com/github/Thilac01/Statistical-Learning-e22395/blob/main/Statistical_Learning_Assignement4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Question 1: The Total Variance Illusion (PCA vs. FA Subspace Allocations)

1. **Physical Nature of Sensor 4:** A Uniqueness value ($\varphi^2$) exceeding $98\%$ means that less than $2\%$ of the variance in `Sensor_4` is driven by the shared common factors ($f_1, f_2$). This proves that `Sensor_4` is completely decoupled from the shared structural dynamics of the asset. Physically, this confirms that the variations recorded by `Sensor_4` represent purely localized white noise, such as a major hardware calibration error or an electrical instrumentation short, rather than actual structural behavior.
2. **The PCA Illusion:** PCA operates by maximizing global variance without distinguishing between shared structural variation and localized sensor noise. The variance of `Sensor_4` is exceptionally high ($\sigma^2 \approx 2.0$) due to its noise level. To capture this large total variance, the PCA engine rotates its top principal axes toward `Sensor_4`, which artificially inflates the top eigenvalues ($\lambda_1, \lambda_2$) and absorbs this localized noise directly into the primary "clean" subspace.
3. **Operational Monitoring Risks:** If an engineer relies solely on a PCA monitoring pipeline, this creates a major vulnerability. Because the principal subspace is heavily contaminated by the noise of a single malfunctioning sensor (`Sensor_4`), the Hotelling's $T^2$ boundary will regularly trigger false alarms from harmless electrical spikes. Conversely, true structural failures (which are captured by `Sensor_1` and `Sensor_2`) can be masked or ignored because their variance contribution is small relative to the noise of `Sensor_4`.

---

# **Subspace Diagnostics & Feature De-correlation: Case Analysis**

This notebook evaluates two fundamental methodologies for multi-sensor asset diagnostics:
1. **Principal Component Analysis (PCA):** Maximizes global variance non-parametrically.
2. **Factor Analysis (FA):** Parametrically partitions shared structural modes from localized instrument noise.

---

## **Question 1: The Total Variance Illusion (PCA vs. FA Subspace Allocations)**



* **Physical Nature of Sensor 4:** A Uniqueness value ($\varphi^2 > 98\%$) proves that less than $2\%$ of its variance is driven by the structural processes ($f_1, f_2$). It is completely decoupled from the system and represents purely localized white noise (e.g., an electrical short or hardware calibration fault).
* **The PCA Illusion:** PCA maximizes total global variance without distinguishing its source. Because `Sensor_4` has a massive noise variance ($\sigma^2 \approx 2.0$), the PCA engine rotates its top principal components directly toward this noise axis, artificially inflating its top eigenvalues and contaminating the "clean" subspace.
* **Operational Monitoring Risks:** Relying solely on a raw PCA pipeline creates high false-alarm rates because harmless electrical spikes instantly trip the Hotelling's $T^2$ boundary. Simultaneously, true structural degradation (tracked by `Sensor_1` and `Sensor_2`) gets masked or ignored because its energy is small compared to the noise floor.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA, FactorAnalysis

# 1. Regenerate assignment data profile
np.random.seed(42)
n_samples = 2500
f1 = np.random.normal(0, 1, n_samples)
f2 = np.random.normal(0, 1, n_samples)

s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples) # High noise floor

df_asset = pd.DataFrame(np.vstack([s1, s2, s3, s4]).T, columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4'])

# Standardize observations
Z = (df_asset - df_asset.mean()) / df_asset.std()

# 2. Compare Variance Attributions
pca = PCA().fit(Z)
fa = FactorAnalysis(n_components=2).fit(Z)

print("--- QUESTION 1 MATHEMATICAL VERIFICATION ---")
print(f"PCA PC1 + PC2 Explained Variance Ratio: {np.sum(pca.explained_variance_ratio_[:2])*100:.2f}%")
print(f"FA Isolated Sensor_4 Uniqueness (Noise Floor): {fa.noise_variance_[3]*100:.2f}%")
print("\nConclusion: PCA captures the noise of Sensor 4 as structural information, while FA isolates it.")

---

## **Question 2: Decoupling Structural Loading via Rotation (FA Subplot 1 vs. PCA Eigenvectors)**



* **Mathematical Mechanics of Varimax:** Traditional PCA forces eigenvectors to follow a rigid mathematical hierarchy ($\lambda_1 > \lambda_2 > \dots$), often spreading mixed, uninterpretable weights across all sensors on the first component. Factor Analysis relaxes this hierarchy and uses an orthogonal **Varimax rotation** to maximize the variance of the squared loadings down each column, pulling weights cleanly toward $0$ or $\pm 1$.
* **Operator Troubleshooting Advantage:** Varimax creates a **Simple Structure** that groups distinct sensors onto independent factors. Instead of a mixed PCA vector, the operator gets an explicit breakdown: `Sensor_1` and `Sensor_2` load onto `Factor 1` (Primary Structural Mode), while `Sensor_3` maps to `Factor 2` (Operational Mode). If `Factor 1` spikes, the operator knows instantly which specific physical components to inspect.

In [ ]:
# Varimax Rotation Utility
def apply_varimax(loadings, max_iter=500, tol=1e-6):
    p, k = loadings.shape
    R = np.eye(k)
    d = 0
    for i in range(max_iter):
        old_d = d
        L = loadings @ R
        alpha = np.diag(np.sum(L**2, axis=0))
        B = loadings.T @ (L**3 - (1.0 / p) * L @ alpha)
        U, S, Vt = np.linalg.svd(B)
        R = U @ Vt
        d = np.sum(S)
        if old_d != 0 and (d - old_d) / old_d < tol: break
    return loadings @ R

print("--- QUESTION 2 MATRIX RECONSTRUCTION ---")
raw_pca_loadings = pca.components_.T[:, :2]
rotated_fa_loadings = apply_varimax(fa.components_.T)

print("\n[Raw Unrotated PCA Subspace Weights (PC1 vs PC2)]")
print(pd.DataFrame(raw_pca_loadings, index=df_asset.columns, columns=['PC1', 'PC2']).round(3))

print("\n[Varimax-Rotated FA Subspace Weights (Factor 1 vs Factor 2)]")
print(pd.DataFrame(rotated_fa_loadings, index=df_asset.columns, columns=['Factor 1', 'Factor 2']).round(3))

---

## **Question 3: Determining Subspace Truncation ($k$) using $T^2$ and $Q$ Profiles**



* **Curve Profile Trajectory:** Moving from $k=1$ to $k=2$ causes a sharp, massive drop in the Mean Residual $Q$ statistic, confirming that the second component accounts for genuine physical energy. Transitioning from $k=2$ to $k=3$ results in a completely flat, horizontal "elbow."
* **Identifying True Dimensionality ($k=2$):** The distinct elbow at $k=2$ separates true structured physical process variations from random instrument noise. It confirms that both structural underlying dimensions ($f_1, f_2$) are fully captured within the active subspace.
* **Consequences of Over-fitting ($k=3$):** If an engineer scales the monitoring subspace to $k=3$, **pure localized sensor noise is forced into the "clean" subspace**. This degrades Hotelling's $T^2$ by making it track random instrument drift, while leaving the residual $Q$ monitor under-sensitized and blind to real structural failures.

In [ ]:
print("--- QUESTION 3 METRIC TRUNCATION HIGHLIGHTS ---")
Z_proj = Z.to_numpy() @ pca.components_.T
summary_stack = []

for k in range(1, 4):
    t2 = np.mean(np.sum((Z_proj[:, :k]**2) / pca.explained_variance_[:k], axis=1))
    q = np.mean(np.sum(Z_proj[:, k:]**2, axis=1))
    summary_stack.append({"Cutoff (k)": k, "Mean Hotelling T2": round(t2, 3), "Mean Residual Q": round(q, 3)})

print(pd.DataFrame(summary_stack).to_string(index=False))
print("\nObservation: The residual error (Q) drops drastically at k=2 and plateaus, identifying k=2 as the optimal cutoff.")

---

## **Question 4: Operational Trade-offs in System Health Monitoring**



* **Strategy Comparison:**
  * **PCA Strategy ($T^2 + Q$):** Comprehensive because it monitors both in-subspace scale ($T^2$) and out-of-subspace model violations ($Q$), but highly vulnerable to single-point sensor failures.
  * **FA Strategy (Factor Scores):** Purely isolates and monitors the actual underlying physical driving forces ($f_1, f_2$) while explicitly dropping the sensor noise floor.
* **Robustness Choice:** The **Factor Analysis Strategy** is significantly more robust against sensor calibration loss or electrical failures.

#### **Technical Justification**
FA defines the system matrix as $\mathbf{R} = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$, where $\boldsymbol{\Psi}$ isolates unique sensor noise on the diagonal. Thomson’s score projection maps the data via:

$$\mathbf{F} = \mathbf{Z} \mathbf{R}^{-1} \boldsymbol{\Lambda}_{\text{rotated}}$$

When a sensor breaks down and its variance spikes, its uniqueness parameter ($\varphi^2$) approaches $1.0$. This causes its corresponding entry in $\mathbf{R}^{-1}$ to collapse toward zero. The FA engine automatically down-weights and excludes the broken tracking channel, preventing its noise from bleeding into the factor scores. The plant operator can continue safely tracking the asset's true physical status using the remaining operational sensors.

In [ ]:
print("--- QUESTION 4 MATHEMATICAL VERIFICATION (THOMSON SCORES) ---")
# Extract Inverse Correlation Matrix and verify weight distribution
R_inv = np.linalg.inv((Z.T @ Z) / (n_samples - 1))
thomson_weights = R_inv @ rotated_fa_loadings

print("[Thomson Score Linear Projection Weights Matrix]")
print(pd.DataFrame(thomson_weights, index=df_asset.columns, columns=['Weight F1', 'Weight F2']).round(3))
print("\nResult: Notice how Sensor_4's projection weights collapse close to zero automatically, isolating the channel noise!")